# SQL Practice GeneratorClaude-powered SQL practice with PostgreSQL and MySQL sandbox.**Workflow:** pick dialect + question type → generate problem → fill diagnostic form → write SQL → Test / Run / Submit.

In [1]:
# ── Setup ──
import os, sys, json, uuid
from pathlib import Path
from datetime import datetime
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import pandas as pd

# Load .env if present
try:
    from dotenv import load_dotenv
    load_dotenv(Path(os.getcwd()).parent / '.env')
except Exception:
    pass

# Reload module imports so edits to .py files take effect on cell re-run
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
for mod in ['sql_practice_utils', 'sandbox']:
    if mod in sys.modules:
        del sys.modules[mod]
import sql_practice_utils as spu
import sandbox as sbx

# Paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
GEN_DIR = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'generated_problems')
SOLVED_DIR = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'solved')
SESSIONS_DIR = os.path.join(PROJECT_ROOT, 'data', 'outputs', 'sessions')
for d in [GEN_DIR, SOLVED_DIR, SESSIONS_DIR]:
    os.makedirs(d, exist_ok=True)

# Init Claude
claude_ready = spu.init_claude()

# Check sandboxes
pg_ok, pg_msg = sbx.check_postgres()
my_ok, my_msg = sbx.check_mysql()
print(f"PostgreSQL: {pg_msg}")
print(f"MySQL:      {my_msg}")
if not pg_ok and not my_ok:
    print("\nNo sandbox reachable. Run `docker compose up -d` from the project root.")

# Shared state across cells
STATE = {
    'problem': None,
    'hint_index': 0,
    'last_action': None,
}

Claude ready (claude-sonnet-4-5)
PostgreSQL: Postgres reachable
MySQL:      MySQL reachable


## 1. Pick a problem

Choose dialect, question type, and either generate a new problem or replay one you've solved.

In [2]:
# ── Problem Picker ──

dialect_dd = widgets.Dropdown(
    options=[('PostgreSQL', 'postgresql'), ('MySQL', 'mysql')],
    value='postgresql',
    description='Dialect:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='320px'),
)

qtype_dd = widgets.Dropdown(
    options=[(v['label'], k) for k, v in spu.QUESTION_TYPES.items()],
    value='select_analytical',
    description='Type:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='420px'),
)

source_radio = widgets.RadioButtons(
    options=[('New (generate)', 'new'), ('Solved (replay)', 'solved')],
    value='new',
    description='Source:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='320px'),
)

solved_dd = widgets.Dropdown(
    options=[('— pick a saved problem —', None)],
    description='Saved:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='600px', display='none'),
)

generate_btn = widgets.Button(description='Generate Problem', button_style='primary',
                              layout=widgets.Layout(width='200px', height='34px'))
status_out = widgets.Output()
problem_out = widgets.Output()

def refresh_solved_list(*_):
    items = spu.list_problems(GEN_DIR, dialect=dialect_dd.value, qtype=qtype_dd.value)
    opts = [('— pick a saved problem —', None)] + [
        (f"{p['generated_at'][:16]} · {p['title']}", p['path']) for p in items
    ]
    solved_dd.options = opts
    solved_dd.value = None

def on_source_change(change):
    if change['new'] == 'solved':
        solved_dd.layout.display = 'flex'
        refresh_solved_list()
    else:
        solved_dd.layout.display = 'none'

source_radio.observe(on_source_change, names='value')
dialect_dd.observe(refresh_solved_list, names='value')
qtype_dd.observe(refresh_solved_list, names='value')

def render_problem(p):
    with problem_out:
        clear_output(wait=True)
        title = p.get('title', 'Untitled')
        prompt = p.get('prompt', '')
        ddl = p.get('schema_ddl', '')
        ex_in = p.get('example_input_data', '')
        ex_cols = p.get('example_output_columns', [])
        ex_rows = p.get('example_output_rows', [])
        ex_df = pd.DataFrame(ex_rows, columns=ex_cols)
        meta = p.get('_meta', {})
        html = f'''
        <div style="border:1px solid #d0d7de; border-radius:6px; padding:16px; background:#fafbfc;">
          <div style="font-size:11px; color:#57606a; margin-bottom:8px;">{meta.get('dialect','')} · {meta.get('question_type','')} · id {meta.get('problem_id','')}</div>
          <h3 style="margin:0 0 12px;">{title}</h3>
          <p style="line-height:1.6;">{prompt}</p>
          <h4 style="margin-top:18px;">Schema</h4>
          <pre style="background:#0d1117; color:#e6edf3; padding:10px; border-radius:4px; overflow-x:auto; font-size:12px;">{ddl}</pre>
          <h4 style="margin-top:18px;">Example input data</h4>
          <pre style="background:#0d1117; color:#e6edf3; padding:10px; border-radius:4px; overflow-x:auto; font-size:12px;">{ex_in}</pre>
          <h4 style="margin-top:18px;">Expected output (example data)</h4>
          {ex_df.to_html(index=False, classes='ex-out')}
        </div>
        '''
        display(HTML(html))

def _attempt_progress(attempt, total, last_error):
    with status_out:
        if last_error:
            print(f'Attempt {attempt-1}/{total} failed validation: {last_error[:200]}')
            print(f'Retrying (attempt {attempt}/{total}) ...')
        else:
            print(f'Generating new {qtype_dd.value} problem in {dialect_dd.value} (attempt {attempt}/{total}) ...')

def on_generate(b):
    with status_out:
        clear_output(wait=True)
        if source_radio.value == 'solved':
            path = solved_dd.value
            if not path:
                print('Pick a saved problem first.')
                return
            print('Loading saved problem ...')
            problem = spu.load_problem(path)
        else:
            problem = spu.generate_problem(
                qtype_dd.value, dialect_dd.value, on_attempt=_attempt_progress
            )
            if not problem:
                print('Generation failed validation. Click Generate to try again.')
                return
            saved = spu.save_problem(problem, GEN_DIR)
            print(f'Validated and saved: {os.path.basename(saved)}')
            attempts = problem.get('_meta', {}).get('validation_attempts', 1)
            print(f'(passed validation on attempt {attempts})')
        STATE['problem'] = problem
        STATE['hint_index'] = 0
        # Refresh the problem reminder in the SQL editor section if it's been created
        try:
            refresh_reminder()
        except NameError:
            pass
        # Load schema and example data into the sandbox for the user to start with
        try:
            sbx.reset(dialect_dd.value)
            sbx.execute_script(dialect_dd.value, problem.get('schema_ddl', ''))
            sbx.execute_script(dialect_dd.value, problem.get('example_input_data', ''))
            print('Sandbox loaded with example data.')
        except Exception as e:
            print(f'Sandbox load error: {e}')
        render_problem(problem)

generate_btn.on_click(on_generate)

display(widgets.VBox([
    widgets.HBox([dialect_dd, qtype_dd]),
    source_radio,
    solved_dd,
    generate_btn,
    status_out,
    problem_out,
]))

## 2. Diagnose before coding

Fill out the form, then click Get Feedback to have your problem analysis graded. Skip if you want to jump straight to coding.

In [7]:
# ── Diagnostic Form ──

paraphrase_ta = widgets.Textarea(
    placeholder='Restate the prompt in your own words ...',
    layout=widgets.Layout(width='100%', height='70px'),
    description='Paraphrase:',
    style={'description_width': '110px'},
)

input_dd = widgets.Dropdown(
    options=[('— pick —', ''), ('Single table', 'single_table'), ('Join 2+', 'join'),
             ('Union', 'union'), ('Procedural', 'procedural')],
    description='Input:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='400px'),
)

output_dd = widgets.Dropdown(
    options=[('— pick —', ''), ('Fewer rows', 'fewer_rows'), ('Same rows', 'same_rows'),
             ('Single value', 'single_value'), ('State mutation', 'state_mutation'),
             ('Scalar return', 'scalar_return'), ('Table return', 'table_return')],
    description='Output shape:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='400px'),
)

recipe_dd = widgets.Dropdown(
    options=[('— pick —', '')] + [(r, r) for r in spu.RECIPE_VOCAB],
    description='Recipe:',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='400px'),
)

moves_ta = widgets.Textarea(
    placeholder='Move 1: ...\nMove 2: ...',
    layout=widgets.Layout(width='100%', height='80px'),
    description='Moves:',
    style={'description_width': '110px'},
)

feedback_btn = widgets.Button(description='Get Feedback', button_style='info',
                              layout=widgets.Layout(width='180px', height='34px'))
feedback_out = widgets.Output()

def fmt_check(ok, label, msg):
    badge = ('<span style="color:#1a7f37; font-weight:600;">correct</span>' if ok
             else '<span style="color:#cf222e; font-weight:600;">rethink</span>')
    return f'<li><strong>{label}:</strong> {badge} — {msg}</li>'

def on_feedback(b):
    with feedback_out:
        clear_output(wait=True)
        if not STATE.get('problem'):
            print('Generate a problem first.')
            return
        answers = {
            'paraphrase': paraphrase_ta.value,
            'input_arrival': input_dd.value,
            'output_shape': output_dd.value,
            'recipe': recipe_dd.value,
            'composite_moves': moves_ta.value,
        }
        print('Asking Claude to grade ...')
        result = spu.grade_diagnostic(STATE['problem'], answers)
        clear_output(wait=True)
        if not result:
            print('Grading failed.')
            return
        html = '<div style="border:1px solid #d0d7de; border-radius:6px; padding:14px; background:#f6f8fa;">'
        html += '<h4 style="margin:0 0 10px;">Diagnostic Feedback</h4>'
        html += '<ul style="line-height:1.7; margin:0 0 0 18px;">'
        html += f'<li><strong>Paraphrase:</strong> {result.get("paraphrase_feedback","")}</li>'
        html += fmt_check(result.get('input_classification_correct', False), 'Input',
                          result.get('input_classification_feedback',''))
        html += fmt_check(result.get('output_shape_correct', False), 'Output shape',
                          result.get('output_shape_feedback',''))
        html += fmt_check(result.get('recipe_correct', False), 'Recipe',
                          result.get('recipe_feedback',''))
        html += f'<li><strong>Moves:</strong> {result.get("composite_moves_feedback","")}</li>'
        html += '</ul>'
        html += f'<p style="margin:10px 0 0; font-style:italic; color:#57606a;">{result.get("overall","")}</p>'
        html += '</div>'
        display(HTML(html))

feedback_btn.on_click(on_feedback)

display(widgets.VBox([paraphrase_ta, input_dd, output_dd, recipe_dd, moves_ta, feedback_btn, feedback_out]))

## 3. Write your SQL

- **Test**: run against example data, see output (does not check correctness).
- **Run**: compare your output to the expected output for example data.
- **Submit**: run against hidden test data; passing saves to your solved bank.

In [9]:
# ── Code Editor ──

def _render_problem_reminder():
    p = STATE.get('problem')
    if not p:
        return widgets.HTML('<div style="color:#57606a; padding:10px;"><i>Generate a problem first.</i></div>')
    title = p.get('title', 'Untitled')
    prompt = p.get('prompt', '')
    schema = p.get('schema_ddl', '')
    return widgets.HTML(f'''
    <div style="border:1px solid #d0d7de; border-radius:6px; padding:12px 16px; background:#fafbfc; margin-bottom:10px;">
      <div style="font-weight:600; margin-bottom:6px;">{title}</div>
      <div style="line-height:1.6; font-size:13px; margin-bottom:8px;">{prompt}</div>
      <details style="margin-top:6px;"><summary style="cursor:pointer; color:#0969da;">Schema</summary>
      <pre style="background:#0d1117; color:#e6edf3; padding:8px; border-radius:4px; overflow-x:auto; font-size:12px; margin-top:6px;">{schema}</pre>
      </details>
    </div>
    ''')

reminder_box = widgets.VBox([_render_problem_reminder()])

def refresh_reminder():
    reminder_box.children = [_render_problem_reminder()]

code_ta = widgets.Textarea(
    placeholder='-- Write your SQL here',
    layout=widgets.Layout(width='100%', height='240px'),
)

test_btn = widgets.Button(description='Test', button_style='',
                          layout=widgets.Layout(width='110px'))
run_btn  = widgets.Button(description='Run', button_style='info',
                          layout=widgets.Layout(width='110px'))
submit_btn = widgets.Button(description='Submit', button_style='success',
                            layout=widgets.Layout(width='110px'))
hint_btn = widgets.Button(description='Hint', button_style='warning',
                          layout=widgets.Layout(width='110px'))
result_out = widgets.Output()
hint_out = widgets.Output()

def on_hint(b):
    with hint_out:
        clear_output(wait=True)
        if not STATE.get('problem'):
            print('Generate a problem first.')
            return
        idx = STATE.get('hint_index', 0)
        text = spu.get_hint(STATE['problem'], idx)
        STATE['hint_index'] = idx + 1
        total = len(STATE['problem'].get('hints', []))
        display(HTML(f'<div style="background:#fff8c5; border-left:4px solid #d4a72c; padding:10px 14px; border-radius:4px;"><strong>Hint {min(idx+1,total)}/{total}:</strong> {text}</div>'))

hint_btn.on_click(on_hint)

def _current_dialect():
    return STATE.get('problem', {}).get('_meta', {}).get('dialect', 'postgresql')

def _load_example_into_sandbox():
    p = STATE.get('problem')
    d = _current_dialect()
    sbx.reset(d)
    sbx.execute_script(d, p.get('schema_ddl', ''))
    sbx.execute_script(d, p.get('example_input_data', ''))

def _load_test_into_sandbox():
    p = STATE.get('problem')
    d = _current_dialect()
    sbx.reset(d)
    sbx.execute_script(d, p.get('schema_ddl', ''))
    sbx.execute_script(d, p.get('test_data', ''))

def _show_df(df, label='Output'):
    if df is None:
        return
    if df.empty:
        display(HTML(f'<div style="color:#57606a;"><b>{label}:</b> empty result set.</div>'))
    else:
        display(HTML(f'<h4>{label}</h4>' + df.to_html(index=False)))

def on_test(b):
    refresh_reminder()
    with result_out:
        clear_output(wait=True)
        if not STATE.get('problem'):
            print('Generate a problem first.')
            return
        try:
            _load_example_into_sandbox()
        except Exception as e:
            print(f'Sandbox load error: {e}')
            return
        df, err = sbx.run_query(_current_dialect(), code_ta.value)
        if err:
            display(HTML(f'<div style="background:#ffebe9; border-left:4px solid #cf222e; padding:10px;"><b>Error:</b><pre style="white-space:pre-wrap; margin:6px 0 0;">{err}</pre></div>'))
            return
        _show_df(df, 'Test output (example data)')

def on_run(b):
    refresh_reminder()
    with result_out:
        clear_output(wait=True)
        if not STATE.get('problem'):
            print('Generate a problem first.')
            return
        try:
            _load_example_into_sandbox()
        except Exception as e:
            print(f'Sandbox load error: {e}')
            return
        df, err = sbx.run_query(_current_dialect(), code_ta.value)
        if err:
            display(HTML(f'<div style="background:#ffebe9; border-left:4px solid #cf222e; padding:10px;"><b>Error:</b><pre style="white-space:pre-wrap; margin:6px 0 0;">{err}</pre></div>'))
            return
        expected = spu.expected_to_dataframe(STATE['problem'], 'example')
        ok, msg = sbx.compare_results(df, expected)
        color = '#dcfce7' if ok else '#ffebe9'
        bar   = '#1a7f37' if ok else '#cf222e'
        verdict = 'CORRECT on example data' if ok else 'MISMATCH on example data'
        display(HTML(f'<div style="background:{color}; border-left:4px solid {bar}; padding:10px; margin-bottom:10px;"><b>{verdict}</b><pre style="white-space:pre-wrap; margin:6px 0 0;">{msg}</pre></div>'))
        _show_df(df, 'Your output')
        _show_df(expected, 'Expected output')

def on_submit(b):
    refresh_reminder()
    with result_out:
        clear_output(wait=True)
        if not STATE.get('problem'):
            print('Generate a problem first.')
            return
        try:
            _load_test_into_sandbox()
        except Exception as e:
            print(f'Sandbox load error: {e}')
            return
        df, err = sbx.run_query(_current_dialect(), code_ta.value)
        if err:
            display(HTML(f'<div style="background:#ffebe9; border-left:4px solid #cf222e; padding:10px;"><b>Error on hidden test data:</b><pre style="white-space:pre-wrap; margin:6px 0 0;">{err}</pre></div>'))
            return
        expected = spu.expected_to_dataframe(STATE['problem'], 'test')
        ok, msg = sbx.compare_results(df, expected)
        color = '#dcfce7' if ok else '#ffebe9'
        bar   = '#1a7f37' if ok else '#cf222e'
        verdict = 'PASS — saved to solved bank' if ok else 'FAIL on hidden test data'
        if ok:
            try:
                spu.save_solved(STATE['problem'], code_ta.value, SOLVED_DIR)
            except Exception as e:
                msg += f'\n(could not save solved record: {e})'
        display(HTML(f'<div style="background:{color}; border-left:4px solid {bar}; padding:10px; margin-bottom:10px;"><b>{verdict}</b><pre style="white-space:pre-wrap; margin:6px 0 0;">{msg}</pre></div>'))
        if not ok:
            _show_df(df, 'Your output (hidden test data)')
            _show_df(expected, 'Expected output (hidden test data)')

test_btn.on_click(on_test)
run_btn.on_click(on_run)
submit_btn.on_click(on_submit)

display(widgets.VBox([reminder_box, code_ta, widgets.HBox([test_btn, run_btn, submit_btn, hint_btn]), hint_out, result_out]))

## 4. Next problem

Clear all fields and start over.

In [6]:
# ── Next Question ──
next_btn = widgets.Button(description='Next Question (clear all)', button_style='danger',
                          layout=widgets.Layout(width='240px', height='34px'))
next_out = widgets.Output()

def on_next(b):
    STATE['problem'] = None
    STATE['hint_index'] = 0
    code_ta.value = ''
    paraphrase_ta.value = ''
    moves_ta.value = ''
    input_dd.value = ''
    output_dd.value = ''
    recipe_dd.value = ''
    try:
        refresh_reminder()
    except NameError:
        pass
    for area in [problem_out, feedback_out, result_out, hint_out, status_out, next_out]:
        try:
            with area:
                clear_output(wait=True)
        except Exception:
            pass
    with next_out:
        print('Cleared. Generate a new problem above.')

next_btn.on_click(on_next)
display(widgets.VBox([next_btn, next_out]))

---**Files written each run:**- `data/outputs/generated_problems/` — every generated problem (loadable via Source: Solved).- `data/outputs/solved/` — successful submissions with your solution code.**Reset the sandbox** when something gets weird: `docker compose down -v && docker compose up -d`.